In [22]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 150)

MARKET = "eth_cbbtc_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_labelled"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_1h,drawdown_1h,trend_1h,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,hash_short,event_sequence_type
0,0xcbf9186fe08ab70d98098d274e46f380248321e1bc07...,MarketSupply,1757435615,0x0000000000000000000000000000000000000001,1100000,1.099788,0,0.0,eth_cbbtc_usdc,2025-09-09 16:33:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,4.708136e+08,3.763097e+08,4.708139e+08,3.763100e+08,0.799275,0.799275,1,0.041319,0.033026,0.041319,0.033026,111014.000000,0.999803,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.099783,0.0,0.0,0.0,loan_position_supply,False,0,0,0,0.003739,0.0,-0.015004,0.003234,0.000000,-0.012060,0xcbf9186f,loan_position_supply
1,0xc7938048f930dcad275757ef6531bc27cce264d6cead...,MarketSupply,1761595235,0x000000000000000000000000000000000000dEaD,10586,0.010585,0,0.0,eth_cbbtc_usdc,2025-10-27 20:00:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,4.769922e+08,4.168996e+08,4.769930e+08,4.169004e+08,0.874018,0.874018,1,0.044545,0.038934,0.044545,0.038934,114836.091413,0.999801,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010584,0.0,0.0,0.0,loan_position_supply,False,0,0,0,0.004437,0.0,-0.000236,0.003686,-0.012068,0.011874,0xc7938048,loan_position_supply


,datetime,type,total_borrow_before,total_borrow_after,vault_flg
2976,2025-09-20 15:03:35,MarketWithdraw,4.057149e+08,4.057158e+08,False
3000,2025-09-30 19:39:23,MarketWithdraw,3.590746e+08,3.590748e+08,False
3017,2025-10-03 19:37:35,MarketWithdraw,4.267607e+08,4.267608e+08,False
3027,2025-10-05 15:12:11,MarketWithdraw,4.259398e+08,4.259399e+08,False
3057,2025-10-10 18:00:59,MarketWithdraw,4.004634e+08,4.004634e+08,False
...,...,...,...,...,...
59907,2026-01-07 11:33:47,MarketWithdraw,3.949392e+08,3.949392e+08,True
59922,2026-01-07 21:40:47,MarketWithdraw,4.001772e+08,4.001772e+08,True
59951,2026-01-08 08:32:59,MarketWithdraw,4.000759e+08,4.000759e+08,True
59953,2026-01-08 09:40:47,MarketWithdraw,4.000779e+08,4.000780e+08,True


In [12]:
def detect_utilization_shocks(hourly_df, threshold=0.1, cutoff_date=None):
    df = hourly_df.sort_values('timestamp').copy()
    df['utilization_change'] = df['utilization'].diff()
    df['utilization_change_abs'] = df['utilization_change'].abs()
    df['is_shock'] = df['utilization_change_abs'] > threshold
    
    if cutoff_date:
        cutoff_ts = int(pd.Timestamp(cutoff_date).timestamp())
        df = df[df['timestamp'] > cutoff_ts]
    
    shock_df = df[df['is_shock']]
    return list(zip(shock_df['timestamp'], shock_df['utilization_change']))
shocks = detect_utilization_shocks(
    market_df,
    cutoff_date="2025-11-01",
)
shocks

[(1761958800, -0.1351587662174918),
 (1762293600, -0.11612628981366202),
 (1762466400, 0.11474145336777308),
 (1767056400, -0.14718934048721644),
 (1767117600, 0.10591637078203686),
 (1767142800, -0.23557347346019586),
 (1767193200, 0.23331363575815378),
 (1767981600, -0.13075056354368286)]

In [28]:
ts = 1762466400
df[
    # (df["utilization_after"] - df["utilization_before"] > 0.01) &
    (df["timestamp"] > ts - 2*3600) &
    (df["timestamp"] < ts + 6*3600)
][[
    "datetime",
    "hash_short",
    "user_address",
    "type",
    "utilization_before",
    "utilization_after",
    "vault_flg",
    "borrow_rate_after",
    "debt_after",
    "supply_after"
]].sort_values("datetime")[:90]

,datetime,hash_short,user_address,type,utilization_before,utilization_after,vault_flg,borrow_rate_after,debt_after,supply_after
12103,2025-11-06 20:03:23,0x4154a5a6,0x7204B7Dbf9412567835633B6F00C3Edc3a8D6330,MarketWithdraw,0.846240,0.846240,True,0.041763,0.000000e+00,-2.426975e+04
47140,2025-11-06 20:03:59,0x8110bf64,0xBEefb9f61CC44895d8AEc381373555a64191A9c4,MarketWithdraw,0.846240,0.857396,True,0.042170,0.000000e+00,-5.118589e+05
47141,2025-11-06 20:03:59,0x8110bf64,0xBEefb9f61CC44895d8AEc381373555a64191A9c4,MarketSupply,0.846240,0.857396,True,0.042170,0.000000e+00,-5.118589e+05
39339,2025-11-06 20:11:59,0x44a42c62,0xBEeFFF209270748ddd194831b3fa287a5386f5bC,MarketSupply,0.857396,0.857396,True,0.042170,0.000000e+00,1.297028e+06
39340,2025-11-06 20:13:35,0x11d8cf80,0xBEeFFF209270748ddd194831b3fa287a5386f5bC,MarketSupply,0.857396,0.857389,True,0.042170,0.000000e+00,1.301079e+06
39341,2025-11-06 20:16:59,0x1f662ee3,0xBEeFFF209270748ddd194831b3fa287a5386f5bC,MarketSupply,0.857389,0.856505,True,0.042137,0.000000e+00,1.798107e+06
18921,2025-11-06 20:22:35,0xd5e22598,0xBA10248Ac4DCC76ef27b58Aad637F02A27FBc4f6,MarketWithdrawCollateral,0.856507,0.856507,False,0.042137,5.648965e+05,0.000000e+00
16746,2025-11-06 20:22:35,0x420163b4,0x974c8FBf4fd795F66B85B73ebC988A51F1A040a9,MarketWithdraw,0.856505,0.856507,True,0.042137,0.000000e+00,2.807137e+07
8265,2025-11-06 20:24:11,0xd5d51902,0x62fE596d59fB077c2Df736dF212E0AFfb522dC78,MarketSupply,0.856507,0.855265,True,0.042092,0.000000e+00,6.991992e+05
47142,2025-11-06 20:29:11,0xf9db4f8b,0xBEefb9f61CC44895d8AEc381373555a64191A9c4,MarketSupply,0.855265,0.855259,True,0.042092,0.000000e+00,-5.085000e+05


In [ ]:
addrs = [
    "0xbCd16D361adf5E7e0dc0010756B49666b4f38eE0",
    "0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB"
]